In [ ]:
# ── Cell 1:import ──
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.gesture_demo.model import GestureMLP
from src.gesture_demo.dataset import GestureDataset
from src.gesture_demo.session_split import session_train_test_split

In [ ]:
# ── Cell 2:讀資料 + 切分 ──
cols = ["label", "session_id"] + [f"{ax}{i}" for i in range(21) for ax in ["x", "y", "z"]]
df = pd.read_csv("data/raw/gestures.csv", header=None, names=cols)
train_df, test_df = session_train_test_split(df, test_ratio=0.25, seed=42)

train_ds = GestureDataset(train_df)
test_ds = GestureDataset(test_df)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

In [ ]:
# ── Cell 3:建模型 ──
model = GestureMLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
# ── Cell 4:訓練 ──
for epoch in range(30):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch+1:2d}: loss = {total_loss/len(train_loader):.4f}")

In [ ]:
# ── Cell 5:評估 ──
from sklearn.metrics import confusion_matrix, classification_report
from src.gesture_demo.dataset import GESTURE_LABELS

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        idx = model(X_batch).argmax(dim=1)
        all_preds.extend(idx.tolist())
        all_labels.extend(y_batch.tolist())

acc = sum(p==l for p,l in zip(all_preds, all_labels)) / len(all_labels)
print(f"test accuracy: {acc:.4f}")
print(classification_report(all_labels, all_preds, target_names=GESTURE_LABELS))


In [ ]:
# ── Cell 6:存模型(想存才跑)──
import os
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/gesture_mlp.pth")
print("已存")